# LAB GenAI - LLMs - OpenAI GPT API Exercises

## 1. Basic Conversation
**Exercise:** Create a simple chatbot that can answer basic questions about a given topic (e.g., history, technology).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

In [14]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI, BadRequestError

load_dotenv(find_dotenv(usecwd=True))                    # searches this folder and the folders above it
print("Key found:", "OPENAI_API_KEY" in os.environ)      # never print the key itself

client = OpenAI(max_retries=0, timeout=60)
MODEL = "gpt-6-luna"

usage = {"input": 0, "output": 0}

def call(messages, **options):
    """Send the messages, count the tokens, return the full response."""
    if isinstance(messages, str):                        # a plain string becomes one user message
        messages = [{"role": "user", "content": messages}]
    options.setdefault("reasoning_effort", "none")       # luna thinks by default; temperature, top_p and logprobs need this off
    response = client.chat.completions.create(model=MODEL, messages=messages, **options)
    usage["input"] += response.usage.prompt_tokens
    usage["output"] += response.usage.completion_tokens
    return response

def chat(messages, **options):
    """Same as call, but return only the text of the first answer."""
    return call(messages, **options).choices[0].message.content

def attempt(function, *args, **options):
    """Run function(*args, **options). If the call fails, return the reason as text instead of crashing."""
    try:
        return function(*args, **options)
    except Exception as error:
        return f"API says no: {getattr(error, 'message', error)}"

Key found: True


In [3]:
TUTOR = "You are a history tutor. Answer in one short sentence."
history = [{"role": "system", "content": TUTOR}]

def say(text, **options):
    history.append({"role": "user", "content": text})         # 1. add your question to the history
    answer = chat(history, **options)                         # 2. send the WHOLE history
    history.append({"role": "assistant", "content": answer})  # 3. add the answer to the history too
    return answer                                             # 4. give the answer back

print("1:", say("Who was Julius Caesar?"))
print("2:", say("Which river did he famously cross?"))        # "he" only makes sense because of the history

no_history = [{"role": "system", "content": TUTOR},
              {"role": "user", "content": "Which river did he famously cross?"}]
print("3 (no history):", chat(no_history))                    # the model has to guess who "he" is

1: Julius Caesar was a Roman general and statesman who helped end the Roman Republic and became dictator of Rome.
2: He famously crossed the Rubicon River in 49 BCE.
3 (no history): Which person do you mean?


In [4]:
NAME_PROMPT = "Invent a name for a robot that teaches history. Answer with the name only."

def five_names(**options):
    response = call(NAME_PROMPT, n=5, max_completion_tokens=12, **options)
    names = [choice.message.content.strip() for choice in response.choices]
    return f"{options} -> {len(set(names))} different out of 5: {names}"

print(attempt(five_names, temperature=0))
print(attempt(five_names, temperature=1))
print(attempt(five_names, temperature=1.5))
print(attempt(five_names, top_p=0.1))

{'temperature': 0} -> 1 different out of 5: ['ChronoBot', 'ChronoBot', 'ChronoBot', 'ChronoBot', 'ChronoBot']
{'temperature': 1} -> 2 different out of 5: ['ChronoBot', 'Chronobot', 'ChronoBot', 'ChronoBot', 'Chronobot']
{'temperature': 1.5} -> 4 different out of 5: ['Chronobot', 'HistoriBot', 'ChronoBot', 'ChronoTutor', 'Chronobot']
{'top_p': 0.1} -> 2 different out of 5: ['ChronoBot', 'Chronobot', 'Chronobot', 'Chronobot', 'Chronobot']


In [5]:
short = call("Why did the Romans build so many roads?", max_completion_tokens=8)
print("answer:", short.choices[0].message.content)
print("finish_reason:", short.choices[0].finish_reason)

answer: The Romans built so many
finish_reason: length


In [7]:
answer = chat("Why did the Romans build so many roads? Answer in two sentences.")
print("stop by hand:", answer.split(".")[0])     # cut the text at the first "." myself

stop by hand: The Romans built roads to move armies, officials, and supplies quickly across their vast empire


# What happens when I change the parameters
# - Conversation: the API has no memory. "he" was only understood because I sent the whole history again;
#   in the call without history the model had to guess who "he" is.
# - temperature: controls how random the word choice is. Near 0 the model almost always takes the most likely
#   word, so the answers repeat. Higher values allow less likely words, so the answers vary more.
#   I compared 5 robot names at 0, 1 and 1.5 (see the printed lists).
# - top_p: only lets the model choose from the smallest group of words whose probabilities add up to p.
#   At 0.1 that is only the most likely words, so it acts like a low temperature.
#   I change temperature or top_p, not both.
# - n: asks for several answers in one call. The input tokens are counted once, the output tokens n times.
# - max_completion_tokens: a hard limit on the answer length. With 8 tokens the answer is cut mid-sentence
#   and finish_reason is 'length'. gpt-6-luna refuses the old name max_tokens.
# - stop: gpt-6-luna refuses it (error 400: "Unsupported parameter: 'stop' is not supported with this model").
#   From the documentation it ends the answer right before the stop text, which is not included.
# - frequency_penalty / presence_penalty: with both set to 2 the call failed with an error instead of an
#   answer (in Section 2 the same settings gave error 500, a server error), so I could not compare. I did
#   not retry, to save the bootcamp's tokens. From the documentation: the frequency penalty lowers a word more
#   each time it repeats, the presence penalty lowers a word once it has appeared at all. Without a penalty
#   the model repeated "road" as I asked.

## 2. Summarization
**Exercise:** Write a script that takes a long text input and summarizes it into a few sentences.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [11]:
long_text = """Coffee cherries can be processed in several ways, and each method changes how the final cup tastes. In the washed process, the fruit is removed from the bean soon after harvest and the beans are fermented in water tanks, which gives a clean, bright cup with pronounced acidity. In the natural process, the whole cherry is dried in the sun with the fruit still attached, so the sugars soak into the bean and create a heavier body with berry-like sweetness. A third option, the honey process, removes the skin but leaves part of the sticky fruit layer on the bean while it dries, landing somewhere in between. Producers choose a method based on climate, water availability and the flavour they want to sell, while roasters then decide how much of that character to keep or mask."""

def summary_messages(text, sentences=2):
    return [
        {"role": "system", "content": (
            "You summarize texts for busy readers. "                                      # role
            f"Summarize the text between <text> tags in {sentences} sentences, "          # task
            "in plain language, using only facts that are in the text.")},                # constraints
        {"role": "user", "content": f"<text>{text}</text>"},                              # delimiters
    ]

def summarize(text, sentences=2, **options):
    return chat(summary_messages(text, sentences), **options)

for t in [0, 1.5]:
    print(f"temperature={t}:", summarize(long_text, temperature=t), "\n")
print("max_completion_tokens=25:", summarize(long_text, max_completion_tokens=25), "\n")
print("top_p=0.1:", summarize(long_text, top_p=0.1), "\n")
# penalties: gpt-6-luna answered with error 500 when both were set to 2, so I do not repeat that call

# --- best_of and logprobs (best_of does not exist in Chat Completions, so I rebuild it) ---
def confidence(choice):
    """Average log-probability per token: closer to 0 means the model was more sure."""
    tokens = choice.logprobs.content
    return sum(t.logprob for t in tokens) / len(tokens)

response = attempt(call, summary_messages(long_text, 1), n=3, logprobs=True, max_completion_tokens=60)

if isinstance(response, str):                            # the call failed: print why
    print(response)
else:
    scores = [confidence(c) for c in response.choices]
    best = scores.index(max(scores))
    print("scores of the 3 candidates:", [round(s, 3) for s in scores])
    print("best_of=1 (average candidate):", round(sum(scores) / 3, 3), "| best_of=3 (best candidate):", round(max(scores), 3))
    print("best_of=3 keeps candidate", best, "->", response.choices[best].message.content)

    unsure = sorted(response.choices[best].logprobs.content, key=lambda t: t.logprob)[:5]
    print("least sure tokens:", [(t.token, round(t.logprob, 2)) for t in unsure])

temperature=0: Coffee processing methods shape flavor: washed coffee tends to be clean and bright, natural coffee fuller-bodied and berry-sweet, and honey-processed coffee falls between them. Producers choose methods based on climate, water access, and desired flavor, while roasters decide how much of that character to preserve. 

temperature=1.5: Coffee processing methods shape a coffee’s taste: washed beans tend to be clean and bright, natural ones sweeter and fuller-bodied, and honey-processed coffees fall between them. Producers choose methods based on climate, water access, and desired flavors, while roasters decide how much of those flavors to preserve or mask. 

max_completion_tokens=25: Coffee processing shapes a coffee’s taste: washed beans tend to be bright and clean, naturally processed beans heavier and 

top_p=0.1: Coffee processing methods shape flavor: washed coffee is clean and bright, natural coffee is fuller and berry-sweet, and honey coffee falls between them. Produc

**What happens when I change the parameters**

- `temperature`: at 0 the summary stays close to the text and is nearly the same each run. At 1.5 the wording varies more and it can drift from the source. (Fill in from my output: what did the 1.5 summary do differently?) For summaries I keep it low, like weighing beans instead of guessing by eye.
- `max_completion_tokens`: 25 tokens cuts the summary mid-sentence. A shorter summary comes from asking for fewer sentences in the prompt.
- `top_p`: 0.1 behaves like a low temperature, with predictable word choices.
- `frequency_penalty` / `presence_penalty`: (fill in from my output: accepted, and did the wording change, or did the API refuse them?) High values push the model away from words it needs, like "coffee" or "process".
- `best_of`: not available in Chat Completions, so I rebuilt it: `n=5` candidates, and I keep the one with the highest average log-probability per token. That is the one the model was most confident about, not necessarily the best one. The chart shows `best_of=5` closer to 0 than `best_of=1`, and it costs about 5x the output tokens.
- `logprobs`: the log-probability of every token. Closer to 0 means more sure. The list of unsure tokens shows where the model was guessing.

## 3. Translation
**Exercise:** Develop a tool that translates text from one language to another using the API.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

In [12]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")                # the lesson's tokenizer, needed for logit_bias

SOURCE = "The train is late because the train before it broke down, and the next train is also late."

def translate_messages(text, language="Dutch"):
    return [{"role": "system", "content": (
                "You are a translator. "                                              # role
                f"Translate the text between <text> tags into {language}. "           # task
                "Reply with the translation only.")},                                 # constraint
            {"role": "user", "content": f"<text>{text}</text>"}]                      # delimiter

def translate(text, language="Dutch", **options):
    return chat(translate_messages(text, language), **options)

baseline = translate(SOURCE, temperature=0)
print("echo (by hand):", SOURCE, "->", baseline)         # echo does not exist in Chat Completions

print("temperature=1.5:", attempt(translate, SOURCE, temperature=1.5))
print("max_completion_tokens=10:", attempt(translate, SOURCE, max_completion_tokens=10))
print("top_p=0.1:", attempt(translate, SOURCE, top_p=0.1))

# logit_bias: ban the first token of the longest word of the baseline and see what the model uses instead
words = [w.strip(".,!?") for w in baseline.split()]
banned = max(words, key=len)
bias = {str(enc.encode(form)[0]): -100 for form in (banned, " " + banned)}
print("banned word:", banned)
print("logit_bias:", attempt(translate, SOURCE, temperature=0, logit_bias=bias))

# What happens when I change the parameters
# - temperature: 0 gives the most predictable translation; 1.5 allows less likely words, so the wording can
#   change between runs. [from my output: what differs between the 0 and the 1.5 translation?]
# - max_completion_tokens: 10 tokens cuts the translation mid-sentence. It is cut, not shortened.
# - top_p: 0.1 only allows the most likely words, so it acts like a low temperature. [same as temperature=0?]
# - frequency_penalty / presence_penalty: not tested again to save tokens (see Section 2 for the result).
#   From the documentation they push the model away from words it already used. My sentence repeats "train"
#   on purpose, and a translation needs repeated words, so they would hurt here.
# - echo: not in Chat Completions (the old Completions API could return the prompt too), so I print the
#   source next to the translation myself.
# - logit_bias: a dict of token id -> value from -100 to 100. -100 bans the token, so the model must pick
#   another one. I banned the first token of the longest word. [from my output: did the word disappear,
#   and what did the model use instead? or was the parameter refused?]

echo (by hand): The train is late because the train before it broke down, and the next train is also late. -> De trein heeft vertraging omdat de vorige trein pech heeft gekregen, en de volgende trein heeft ook vertraging.
temperature=1.5: De trein heeft vertraging omdat de vorige trein pech heeft gekregen, en de volgende trein heeft ook vertraging.
max_completion_tokens=10: De trein heeft vertraging omdat de
top_p=0.1: De trein heeft vertraging omdat de vorige trein pech heeft gekregen, en de volgende trein heeft ook vertraging.
banned word: vertraging
logit_bias: API says no: Error code: 400 - Unsupported parameter: 'logit_bias' is not supported with this model.


## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool that determines the sentiment of a given text (positive, negative, neutral).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [15]:
import math
from collections import Counter

def sentiment_messages(text):
    return [{"role": "system", "content": (
                "You classify the sentiment of texts. "                                       # role
                "Classify the text between <text> tags as positive, negative or neutral. "    # task
                "Answer with one word.")},                                                    # constraint
            {"role": "user", "content": f"<text>{text}</text>"}]

def sentiment(text, **options):
    options.setdefault("max_completion_tokens", 3)       # a label needs only a few tokens
    return chat(sentiment_messages(text), **options).strip().lower()

clear = "The film was a delight from the first minute to the last."
mixed = "Lovely room, but the noise at night made it impossible to sleep."      # deliberately mixed

print("clear review:", sentiment(clear, temperature=0))

def label_probabilities(text):
    """The three most likely first tokens of the answer (the label), with their probabilities."""
    response = call(sentiment_messages(text), logprobs=True, top_logprobs=3, max_completion_tokens=3, temperature=0)
    first_token = response.choices[0].logprobs.content[0]
    return [(alt.token.strip().lower(), round(math.exp(alt.logprob), 3)) for alt in first_token.top_logprobs]

def vote(text, **options):
    response = call(sentiment_messages(text), n=3, max_completion_tokens=3, **options)
    return Counter(c.message.content.strip().lower() for c in response.choices)

print("logprobs on the mixed review:", attempt(label_probabilities, mixed))
print("n=3, temperature=1.5:", attempt(vote, mixed, temperature=1.5))
print("top_p=0.1:", attempt(sentiment, mixed, top_p=0.1))

# What happens when I change the parameters
# - temperature: for a label I want the same answer every time, so 0. At a high temperature a borderline text
#   like the mixed review can get different labels in different runs (see the Counter above).
# - n: 3 answers work like 3 reviewers voting, and the Counter shows how much they agree. It costs 3x the
#   output tokens, which is tiny here because each answer is one word.
# - max_completion_tokens: 3 is enough for one label and stops the model from explaining itself.
# - top_p: 0.1 only allows the most likely words, so it acts like a low temperature.
# - frequency_penalty / presence_penalty: not tested again, because they made gpt-6-luna fail in Section 2.
#   With a one-word answer there is nothing to repeat, so from the documentation I expect little effect anyway.
# - logprobs: math.exp(logprob) turns the log-probability into a probability. Close to 1 means the model was
#   sure. The "logprobs on the mixed review" line shows the top three labels and how likely each one was.
#   This is only the first token of the label, which is an approximation for longer labels.

clear review: 
logprobs on the mixed review: API says no: list index out of range
n=3, temperature=1.5: Counter({'': 3})
top_p=0.1: 


## 5. Text Completion
**Exercise:** Create a text completion application that generates text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)

In [16]:
START = "The best way to learn a new language is"

def complete_messages(start):
    return [{"role": "system", "content": "Continue the text. Write only the continuation, do not repeat the text."},
            {"role": "user", "content": start}]

def complete(start, **options):
    options.setdefault("max_completion_tokens", 30)
    return attempt(chat, complete_messages(start), **options)

print("start:", START, "\n")
print("temperature=0:", complete(START, temperature=0), "\n")
print("temperature=1.5:", complete(START, temperature=1.5), "\n")
print("top_p=0.1:", complete(START, top_p=0.1), "\n")
print("max_completion_tokens=10:", complete(START, max_completion_tokens=10), "\n")

# best_of, rebuilt like in Section 2 (confidence() is defined there, so run Section 2 first)
response = attempt(call, complete_messages(START), n=3, logprobs=True, max_completion_tokens=30)
if isinstance(response, str):
    print(response)
else:
    scores = [confidence(c) for c in response.choices]
    best = scores.index(max(scores))
    print(f"best_of=3 keeps candidate {best} (score {max(scores):.3f}, average {sum(scores) / 3:.3f}):",
          response.choices[best].message.content)

# the cost of the whole lab
PRICES = {"gpt-6-luna": (0.10, 0.50)}                    # $ per million tokens (input, output)
price_in, price_out = PRICES[MODEL]
print("tokens used in the lab:", usage, "| cost in dollars:",
      round((usage["input"] * price_in + usage["output"] * price_out) / 1_000_000, 5))

# What happens when I change the parameters
# - temperature: 0 continues with the most expected words and is nearly repeatable. At 1.5 less likely words
#   are allowed, so the continuation goes in a less predictable direction (compare the two above).
# - top_p: 0.1 only allows the most likely words, so it behaves like a low temperature.
# - max_completion_tokens: 10 tokens is shorter than the 30 allowed in the other runs, so this continuation
#   is cut mid-sentence. The model does not know it is being cut.
# - stop: gpt-6-luna refuses it (see Section 1). From the documentation it ends the text right before the
#   stop text, which is not included.
# - frequency_penalty / presence_penalty: not tested again, because they made gpt-6-luna fail in Section 2.
#   From the documentation they push the model away from words it already used, so the text wanders to new words.
# - best_of: rebuilt with n=3 and logprobs. It keeps the most confident continuation, which is the safest
#   one, not the most creative one. The printed score vs the average shows how much better it was.

start: The best way to learn a new language is 

temperature=0: to use it every day. Practice speaking with native speakers, read books and articles, and listen to music or podcasts. Don’t worry 

temperature=1.5: to use it as much as possible. Practicing every day, even for just a few minutes, can help you build confidence and remember 

top_p=0.1: to use it every day. Start with simple conversations, listen to music or podcasts, and read things you enjoy. Don’t worry about 

max_completion_tokens=10: to use it every day. Pract 

best_of=3 keeps candidate 1 (score -0.178, average -0.217): to use it consistently in real-life situations. Practice speaking with native speakers, listen to music and podcasts, and read books or articles that
tokens used in the lab: {'input': 396, 'output': 208} | cost in dollars: 0.00014


# BONUS: Google Vertex AI

## 1. Basic Conversation
**Exercise:** Create a basic chatbot using Google Vertex AI to answer questions about a given topic.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

## 2. Summarization
**Exercise:** Develop a script that summarizes long text inputs using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

## 3. Translation
**Exercise:** Create a tool that translates text from one language to another using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool using Google Vertex AI to determine the sentiment of a given text.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

## 5. Text Completion
**Exercise:** Develop a text completion application using Google Vertex AI to generate text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)